<img src='https://www.unifor.br/o/unifor-theme/images/unifor-logo-horizontal.svg' width="250px">

# Relatório Semanal de Atividades

Universidade de Fortaleza<br>
Programa de Pós-Graduação em Informática Aplicada (PPGIA)<br>
Laboratório de Ciência de Dados e Inteligência Artificial (LCDIA)

Aluna: Gabriela Ferreira Coutinho<br>
Orientador: Prof. Rilder de Sousa Pires</br>

## Análise da Rede Social Facebook (SNAP - Ego Network)

Este notebook realiza uma análise exploratória do dataset **ego-Facebook** disponível no [Stanford Network Analysis Project (SNAP)](https://snap.stanford.edu/data/ego-Facebook.html).

**Atividades:**
- A1 — Baixar o dataset
- A2 — Carregar o grafo e verificar nós/arestas
- A3 — Calcular grau médio, densidade e distribuição de graus
- A4 — Visualizar um subgrafo pequeno
- A5 — Documentar primeiras observações

## A1 — Baixar o dataset SNAP (ego-Facebook)

In [6]:
import os
import urllib.request
import gzip
import shutil

url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
data_dir = os.path.join("..", "data")
gz_file = os.path.join(data_dir, "facebook_combined.txt.gz")
txt_file = os.path.join(data_dir, "facebook_combined.txt")

os.makedirs(data_dir, exist_ok=True)

if not os.path.exists(txt_file):
    urllib.request.urlretrieve(url, gz_file)
    with gzip.open(gz_file, 'rb') as f_in, open(txt_file, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(gz_file)

os.path.getsize(txt_file) / 1024

834.337890625

## A2 — Carregar o grafo e verificar quantidade de nós/arestas

In [7]:
import networkx as nx

G = nx.read_edgelist(txt_file, nodetype=int)

G.number_of_nodes(), G.number_of_edges(), nx.is_connected(G)

(4039, 88234, True)

## A3 — Calcular grau médio, densidade e distribuição de graus

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

graus = [d for _, d in G.degree()]
grau_medio = sum(graus) / len(graus)
densidade = nx.density(G)

grau_medio, densidade, min(graus), max(graus)

(43.69101262688784, 0.010819963503439287, 1, 1045)

In [9]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Distribuição de Graus', 'Distribuição de Graus (escala log)'))

fig.add_trace(
    go.Histogram(x=graus, nbinsx=50, marker_color='steelblue', name='Linear'),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=graus, nbinsx=50, marker_color='coral', name='Log'),
    row=1, col=2
)

fig.update_xaxes(title_text='Grau', row=1, col=1)
fig.update_yaxes(title_text='Frequência', row=1, col=1)
fig.update_xaxes(title_text='Grau', row=1, col=2)
fig.update_yaxes(title_text='Frequência (log)', type='log', row=1, col=2)

fig.update_layout(height=450, width=900, showlegend=False, template='plotly_white')
fig.show()

## A4 — Visualizar um subgrafo pequeno

In [10]:
no_central = sorted(G.degree(), key=lambda x: x[1])[len(graus) // 2][0]
vizinhos = list(G.neighbors(no_central))
subgrafo = G.subgraph([no_central] + vizinhos)

print(f"Nó central: {no_central} (grau {G.degree(no_central)})")
print(f"Subgrafo: {subgrafo.number_of_nodes()} nós, {subgrafo.number_of_edges()} arestas")

pos = nx.spring_layout(subgrafo, seed=42)

edge_x, edge_y = [], []
for u, v in subgrafo.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.5, color='gray'),
    hoverinfo='none'
)

node_x = [pos[n][0] for n in subgrafo.nodes()]
node_y = [pos[n][1] for n in subgrafo.nodes()]
node_colors = ['red' if n == no_central else 'lightblue' for n in subgrafo.nodes()]
node_sizes = [20 if n == no_central else 10 for n in subgrafo.nodes()]
node_labels = [str(n) for n in subgrafo.nodes()]

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    marker=dict(size=node_sizes, color=node_colors, line=dict(width=1, color='black')),
    text=node_labels, textposition='top center', textfont=dict(size=8),
    hoverinfo='text'
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title=f'Ego Network do nó {no_central}',
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=650, width=650,
    template='plotly_white'
)
fig.show()

Nó central: 3943 (grau 25)
Subgrafo: 26 nós, 251 arestas
